# PowerFlow ENTSO-E training in Google Colab

This notebook prepares a Colab environment and invokes the repository's existing four-model trainer. It does not duplicate model logic or change the chronological 80/20 split. Google Drive is optional.

## 1. Configure the run

Set a supplied gold CSV path. When Drive is enabled, the default conventions follow the documented `PowerFlow/` layout.

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/ianwambaire/Electricity_Trading_Pipeline.git"
REPOSITORY_BRANCH = "main"
REPOSITORY_DIR = Path("/content/electricity-trading-pipeline")

USE_GOOGLE_DRIVE = False
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/PowerFlow")
RELEASE_NAME = "entsoe-baseline"

# Set this to any supplied gold CSV. Leave as None to use the conventional path.
SUPPLIED_GOLD_DATA_PATH = None
MLFLOW_TRACKING_URI = f"sqlite:///{REPOSITORY_DIR / 'mlflow.db'}"

## 2. Clone or update PowerFlow

In [ ]:
import subprocess

if (REPOSITORY_DIR / '.git').exists():
    subprocess.run(
        ['git', '-C', str(REPOSITORY_DIR), 'pull', '--ff-only'],
        check=True,
    )
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPOSITORY_BRANCH, REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

## 3. Install repository dependencies

In [ ]:
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPOSITORY_DIR / 'requirements.txt')],
    check=True,
)

## 4. Optionally mount Google Drive

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive')

## 5. Resolve and validate inputs and outputs

In [ ]:
if SUPPLIED_GOLD_DATA_PATH:
    gold_data_path = Path(SUPPLIED_GOLD_DATA_PATH)
elif USE_GOOGLE_DRIVE:
    gold_data_path = DRIVE_PROJECT_ROOT / 'datasets' / 'snapshots' / 'gold_model_features.csv'
else:
    gold_data_path = REPOSITORY_DIR / 'data' / 'features' / 'gold_model_features.csv'

if USE_GOOGLE_DRIVE:
    release_dir = DRIVE_PROJECT_ROOT / 'models' / 'releases' / RELEASE_NAME
else:
    release_dir = REPOSITORY_DIR / 'artifacts' / 'models'

comparison_path = release_dir / 'gold_model_comparison.csv'
release_dir.mkdir(parents=True, exist_ok=True)

if not gold_data_path.exists():
    raise FileNotFoundError(
        f'Gold dataset not found: {gold_data_path}. Upload it or update SUPPLIED_GOLD_DATA_PATH.'
    )

print('Gold dataset:', gold_data_path)
print('Release output:', release_dir)
print('MLflow tracking URI:', MLFLOW_TRACKING_URI)

## 6. Run the existing PowerFlow trainer

In [ ]:
import os

environment = os.environ.copy()
environment['PYTHONPATH'] = str(REPOSITORY_DIR / 'src')

subprocess.run(
    [
        sys.executable,
        'src/models/train_gold_model.py',
        '--data-path', str(gold_data_path),
        '--output-dir', str(release_dir),
        '--comparison-path', str(comparison_path),
        '--mlflow-tracking-uri', MLFLOW_TRACKING_URI,
    ],
    cwd=REPOSITORY_DIR,
    env=environment,
    check=True,
)

## 7. Inspect the selected model manifest

In [ ]:
import json

manifest_path = release_dir / 'training_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
manifest